## Test Runs on Train Gen. Methods

In [3]:
import os
import sys
import json
import subprocess
import gc
import torch
from pathlib import Path
from collections import Counter


In [2]:
CONFIG = {
    "colab": True,
    "branch": "main",

    "repo_name": "BigDataAndTextMiningProject",

    "repo_owner": "Aivon99",
    "repo_dir": "/content",

}




In [9]:
if CONFIG["colab"]:
    os.makedirs(CONFIG["repo_dir"], exist_ok=True)
    os.chdir("/content")
    repo_dir = Path(CONFIG["repo_dir"])
    if repo_dir.exists():
        subprocess.run(["rm", "-rf", str(repo_dir)], check=True)

    auth_url =  "https://"
    repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

    result = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_dir)],
        capture_output=True, text=True
    )
    assert result.returncode == 0, f"Git clone failed: {result.stderr}"

    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
else:
    repo_dir = Path(".").resolve()
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))

REPO_ROOT = Path(".").resolve()

print("Setup Complete. REPO_ROOT:", REPO_ROOT)

AssertionError: Git clone failed: Cloning into '/content'...
fatal: Unable to read current working directory: No such file or directory


In [1]:
# Cell 1
# Basic imports

from pathlib import Path

from src.data_util.generation import (
    build_sample,
    generate_dataset,
    ChessDatasetGenerator
)

from src.data_util.utilities import (
    load_lichess_csv
)

ModuleNotFoundError: No module named 'src'

In [ ]:
# Cell 2
# Example FEN positions

fens = [

    # Starting position
    "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",

    # Midgame
    "r2qkbnr/ppp1pppp/2n5/3p4/3P1B2/2N4P/PPP1PP2/R2QKBNR w KQkq - 0 1",

    # Endgame
    "8/8/8/2k5/8/5K2/8/8 w - - 0 1"
]

len(fens)

In [ ]:
# Cell 3
# Test single sample generation

sample = build_sample(
    fen=fens[0],
    sample_id="debug_sample",
    image_size=512,
    task="fen"
)

sample["metadata"]

In [ ]:
# Cell 4
# Visualize generated image

sample["image"]

In [ ]:
# Cell 5
# Save one sample to disk

output_dir = Path("debug_dataset")

sample = build_sample(
    fen=fens[1],
    sample_id="saved_example",
    output_dir=output_dir,
    image_size=512,
    task="fen"
)

print("Saved successfully.")

In [ ]:
# Cell 6
# Check saved structure

for path in output_dir.rglob("*"):
    print(path)

In [ ]:
# Cell 7
# Batch dataset generation

generate_dataset(
    fens=fens,
    output_dir="batch_dataset",
    image_size=512,
    task="fen"
)

print("Batch dataset generated.")

In [ ]:
# Cell 8
# Inspect generated dataset

dataset_dir = Path("batch_dataset")

samples = sorted(dataset_dir.iterdir())

print(f"Number of samples: {len(samples)}")

samples[:3]

In [ ]:
# Cell 9
# Open one saved image manually

from PIL import Image

img = Image.open(
    "batch_dataset/sample_000001/board.png"
)

img

In [ ]:
# Cell 10
# Inspect metadata JSON

import json

with open(
    "batch_dataset/sample_000001/metadata.json"
) as f:

    metadata = json.load(f)

metadata

In [ ]:
# Cell 11
# Test generator mode

generator = ChessDatasetGenerator(
    fens=fens,
    image_size=512,
    task="fen"
)

for idx, sample in enumerate(generator):

    print(sample["metadata"]["sample_id"])

    display(sample["image"])

    if idx == 1:
        break

In [ ]:
    # Cell 12
if True:
    # Test loading Lichess CSV

    csv_path = "lichess_db_puzzle.csv"

    df = load_lichess_csv(
        csv_path=csv_path,
        max_samples=5
    )

    df.head()
    # Cell 13
    # Extract FENs from Lichess dataframe

    lichess_fens = df["FEN"].tolist()

    lichess_fens[:3]
    # Cell 14
    # Generate dataset from Lichess sample

    generate_dataset(
        fens=lichess_fens,
        output_dir="lichess_dataset",
        image_size=512,
        task="fen"
    )

    print("Lichess dataset generated.")
    # Cell 15
    # Visualize one generated Lichess sample

    img = Image.open(
        "lichess_dataset/sample_000000/board.png"
    )

    img# Cell 16
    # Inspect metadata of generated Lichess sample

    with open(
        "lichess_dataset/sample_000000/metadata.json"
    ) as f:

        metadata = json.load(f)

    metadata